In [ ]:
import torch
import lightning
import numpy as np
import matplotlib.pyplot as plt
import copy
from mlcolvar.data import DictModule, DictDataset
from mlcolvar.utils.trainer import MetricsCallback
from mlcolvar.utils.plot import plot_metrics
from mlcolvar.core.transform.tools import ContinuousHistogram
from lightning.pytorch.callbacks import EarlyStopping
from mlcolvar.core.transform.descriptors.gnn_transformer import GNNTransformerDescriptor, LightningGNNTransformer
from mlcolvar.core.transform.descriptors.gnn_utils import build_graphs_for_positions, create_normalized_gnn
from mlcolvar.utils.plot import paletteFessa, plot_metrics
import matplotlib.pyplot as plt
from mlcolvar.cvs.committor.utils import compute_committor_weights
from mlcolvar.core.loss.utils.smart_derivatives import compute_descriptors_derivatives
from mlcolvar.cvs.indicator import IndicatorTraining, IndicatorProduction, IndicatorBiasModel


In [2]:
# Helper functions for loading data

def read_all_entropy_energy(filename):
    """
    Reads all timesteps from a LAMMPS dump file containing per-atom entropy and potential energy.

    Parameters:
        filename (str): Path to the LAMMPS dump file.

    Returns:
        entropy_array (np.ndarray): Shape (n_frames, n_atoms), per-atom entropy.
        energy_array  (np.ndarray): Shape (n_frames, n_atoms), per-atom energy.
    """
    entropy_list = []
    energy_list = []

    with open(filename, 'r') as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        # Detect start of a frame
        if "ITEM: TIMESTEP" in lines[i]:
            # Move to number of atoms
            num_atoms = int(lines[i + 3].strip())

            # Move to start of atom lines
            atom_start = i + 9
            atom_end = atom_start + num_atoms

            entropy = []
            energy = []

            for line in lines[atom_start:atom_end]:
                parts = line.strip().split()
                c_ent = float(parts[-2])
                c_peatom = float(parts[-1])
                entropy.append(c_ent)
                energy.append(c_peatom)

            entropy_list.append(entropy)
            energy_list.append(energy)

            # Jump to next frame
            i = atom_end
        else:
            i += 1

    # Convert to numpy arrays
    entropy_array = np.array(entropy_list)
    energy_array = np.array(energy_list)

    return entropy_array, energy_array

def lammps_to_numpy(traj_filename):
    """
    Parse a LAMMPS trajectory file and extract coordinates and per-frame box bounds.

    Parameters:
        traj_filename (str): Path to the LAMMPS trajectory file.

    Returns:
        coords (np.ndarray): Array of shape (n_frames, n_atoms, 3) with atomic positions.
        bounds (List[List[float]]): Per-frame list of [xlo, xhi, ylo, yhi, zlo, zhi].
    """
    coords = []
    bounds = []
    #peatoms = []

    with open(traj_filename, 'r') as traj_file:
        while True:
            line = traj_file.readline()
            if not line:
                break  # End of file

            if "ITEM: TIMESTEP" in line:
                traj_file.readline()  # Skip timestep value

                traj_file.readline()  # ITEM: NUMBER OF ATOMS
                num_atoms = int(traj_file.readline().strip())

                bb_line = traj_file.readline()  # ITEM: BOX BOUNDS ...
                if not bb_line.startswith("ITEM: BOX BOUNDS"):
                    raise ValueError("Expected 'ITEM: BOX BOUNDS' line in trajectory")
                # Read three lines of bounds; each may contain 2 or 3 values (triclinic includes tilt)
                xyz_bounds = []
                for _ in range(3):
                    bline = traj_file.readline().strip().split()
                    if len(bline) < 2:
                        raise ValueError("Malformed BOX BOUNDS line; expected at least two floats")
                    lo = float(bline[0]); hi = float(bline[1])
                    xyz_bounds.append((lo, hi))
                xlo, xhi = xyz_bounds[0]
                ylo, yhi = xyz_bounds[1]
                zlo, zhi = xyz_bounds[2]
                bounds.append([xlo, xhi, ylo, yhi, zlo, zhi])

                atom_header = traj_file.readline().strip()
                if not atom_header.startswith("ITEM: ATOMS"):
                    raise ValueError("Unexpected format in ATOMS section")

                headers = atom_header.split()[2:]
                col_x = headers.index("x")
                col_y = headers.index("y")
                col_z = headers.index("z")
                #col_peatom = headers.index("c_peatom")

                frame_coords = np.zeros((num_atoms, 3), dtype=np.float64)
                #frame_peatoms = np.zeros((num_atoms,), dtype=np.float64)

                for i in range(num_atoms):
                    atom_data = traj_file.readline().strip().split()
                    frame_coords[i, 0] = float(atom_data[col_x])
                    frame_coords[i, 1] = float(atom_data[col_y])
                    frame_coords[i, 2] = float(atom_data[col_z])
                    #frame_peatoms[i] = float(atom_data[col_peatom])

                coords.append(frame_coords)
                #peatoms.append(frame_peatoms)

    return np.array(coords), bounds# , np.array(peatoms)

In [ ]:
# Load data
ent_melt_lammps, e_melt = read_all_entropy_energy("../test_data/just_entropy_test.lammpstrj")

n_at = 559 
traj_melt, bounds_melt = lammps_to_numpy("../test_data/dump.Na.lammpstrj")

In [ ]:
# Create the unbiased dataset
unbiased_data = np.concatenate([traj_melt])
unbiased_labels = np.concatenate([ent_melt_lammps])
hist_ent = ContinuousHistogram(in_features=n_at, min=-6, max=0, bins=40)

histo_labels = hist_ent(torch.Tensor(unbiased_labels.reshape(-1,n_at)))

unbiased_bounds = torch.tensor(np.concatenate([bounds_melt]))

# Create the dataset
ds_unbiased = DictDataset({"data": torch.Tensor(unbiased_data.reshape(-1,n_at*3)), "labels": histo_labels, "bounds": torch.Tensor(unbiased_bounds)})

In [ ]:
# Create the graphs for the dataset

graphs = []
max_edges = 0
for i in range(len(ds_unbiased)):
    pos = ds_unbiased['data'][i]
    graph = build_graphs_for_positions(pos, n_atoms=n_at, PBC=True, cell=unbiased_bounds[i], cutoff=3.75)[0]
    graphs.append(graph)
    max_edges = max(max_edges, graph.shape[1])

for i in range(len(graphs)):
    # Pad with -1 to max_edges
    if graphs[i].shape[1] < max_edges:
        pad_size = max_edges - graphs[i].shape[1]
        pad = torch.full((2, pad_size), -1, dtype=torch.long)
        graph_padded = torch.cat([graphs[i], pad], dim=1)
        graphs[i] = graph_padded

ds_unbiased["graph"] = torch.stack(graphs)

In [ ]:
# Set up training of the graph layer
datamodule = DictModule(ds_unbiased, lengths=[1], shuffle=True, batch_size=32)

# initialise lr scheduler
lr_scheduler = torch.optim.lr_scheduler.ExponentialLR

# create options dictionary
options = {'optimizer' : {'lr': 1e-3, 'weight_decay': 1e-5},
           'lr_scheduler' : {'scheduler' : lr_scheduler, 'gamma' : 0.99999},
            'nn' : {'activation' : 'tanh'}}

# number of descriptors, which is the size of the input layer
n_at = 559

# Compute example cell from first sample's bounds for GNN initialization
# The GNN will receive per-sample bounds at runtime for NPT support
example_bounds = unbiased_bounds[0]  # [xlo, xhi, ylo, yhi, zlo, zhi]
example_cell = [example_bounds[1] - example_bounds[0],  # Lx
                example_bounds[3] - example_bounds[2],  # Ly
                example_bounds[5] - example_bounds[4]]  # Lz

# initialise model
model = GNNTransformerDescriptor(
    n_atoms=n_at,
    out_features=40,
    in_node_nf=3,
    hidden_nf=64,
    n_layers=2,
    n_heads=1,
    PBC=True,
    cell=example_cell,  # Use example cell; actual cell passed at runtime for NPT
    cutoff=3.75, # Match cutoff with cutoff in the entropy calculation
    pool="sum",
    mode="graph",
    device="cuda" if torch.cuda.is_available() else "cpu",
)

metrics = MetricsCallback()


In [19]:
T = 350
kb = 0.0083144621 # kJ/mol
beta = 1/(kb*T)

model_train = LightningGNNTransformer(
    descriptor=model,
    options=options)


# initialize trainer, for testing the number of epochs is low, change this to something like 5000 or 100000
trainer = lightning.Trainer(callbacks=[metrics],
                            max_epochs=100,
                            logger=False,
                            enable_checkpointing=False,
                            limit_val_batches=0,    # this is to skip validation
                            num_sanity_val_steps=0,  # this is to skip validation
                            accelerator='gpu',
                            devices=1,
                            enable_progress_bar=False
                            )

# fit model
model_train = model_train.cuda()  # Move model to GPU
trainer.fit(model_train, datamodule)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name       | Type                     | Params | Mode  | FLOPs
------------------------------------------------------------------------
0 | descriptor | GNNTransformerDescriptor | 82.2 K | train | 0    
1 | criterion  | MSELoss                  | 0      | train | 0    
------------------------------------------------------------------------
82.2 K    Trainable params
0         Non-trainable params
82.2 K    Total params
0.329     Total estimated model params size (MB)
42        Modules in train mode
0         Modules in eval mode
0         Total Flops
/home/fdietrich@iit.local/miniconda3/envs/ensemblecolvar/lib/python3.14/site-packages

In [22]:
labels_melt = torch.zeros(len(ds_unbiased))

# Create the dataset with bounds for NPT support
ds_unbiased = DictDataset({
    "data": torch.Tensor(unbiased_data.reshape(-1, n_at*3)), 
    "labels": torch.tensor(labels_melt),
    "bounds": torch.Tensor(unbiased_bounds)  # Include bounds for NPT
})


/tmp/ipykernel_1633624/1404025176.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "labels": torch.tensor(labels_melt),


In [24]:
# zeroth iteration should be unbiased, we thus initialise the bias as zero
bias = torch.zeros(len(ds_unbiased))

# compute weights
ds_unbiased = compute_committor_weights(dataset=ds_unbiased,
                                    bias=bias,
                                    data_groups=[0],
                                    beta=beta)

model_train.eval()
model_train.freeze()

LightningGNNTransformer(
  (descriptor): GNNTransformerDescriptor(
    (embedding): Linear(in_features=3, out_features=64, bias=True)
    (layers): ModuleList(
      (0-1): 2 x TransGCL(
        (act_fn): ReLU()
        (dropout): Dropout(p=0.1, inplace=False)
        (edge_0): Sequential(
          (0): Linear(in_features=128, out_features=64, bias=True)
          (1): ReLU()
          (2): Linear(in_features=64, out_features=64, bias=True)
        )
        (attention_0): Sequential(
          (0): Linear(in_features=128, out_features=64, bias=True)
          (1): ReLU()
          (2): Linear(in_features=64, out_features=1, bias=True)
        )
        (pre_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
        (node_mlp): Sequential(
          (0): Linear(in_features=128, out_features=64, bias=True)
          (1): ReLU()
          (2): Linear(in_features=64, out_features=64, bias=True)
        )
        (edge_modules): ModuleList(
          (0): Sequential(


In [26]:
# Compute normalization constant over training data
n_samples_for_norm = min(1000, len(ds_unbiased['data']))
positions_for_norm = ds_unbiased['data'][:n_samples_for_norm]

normalized_gnn, mean_jac_norm, std_jac_norm = create_normalized_gnn(
    gnn=model_train,
    positions=positions_for_norm,
    n_atoms=n_at,
    batch_size=32,
    device=next(model_train.parameters()).device,
)

Computing GNN Jacobian Frobenius norms over dataset...
  Mean ||J||_F = 3.097488
  Std  ||J||_F = 0.276488
  Coefficient of variation = 8.93%


In [28]:

device=next(model_train.parameters()).device
pos, desc, d_desc_d_pos = compute_descriptors_derivatives(ds_unbiased, normalized_gnn, n_at, separate_boundary_dataset = False, batch_size=128)
dataset = DictDataset({"data":desc.clone().detach().to(device), "weights":ds_unbiased["weights"].to(device),"derivatives":d_desc_d_pos.clone().detach().to(device)})


Processed all data in 1 batches!


In [29]:
# Store the normalized GNN for later use at export time
model_normalized = normalized_gnn

atomic_masses = torch.ones(559)*22.98976928  # Sodium atomic mass

gamma = 1/0.05
friction = np.zeros(n_at*3)
print(friction.shape)
for i_atom in range(n_at):
    friction[3*i_atom:3*i_atom+3] = np.array([kb*T / (gamma*atomic_masses[i_atom])]*3) 
friction = torch.tensor(friction, device=device,dtype=torch.float32)

(1677,)


In [34]:
gen_options = {"nn": {"activation": "tanh"},
               "optimizer": {"lr": 5e-4, "weight_decay": 1e-5}}

num_descriptors_gen = dataset["data"].shape[1]
gen_layers = [num_descriptors_gen, 32, 32, 1]

dataset_gen = DictDataset({
    "data": dataset["data"].detach().clone(),
    "weights": dataset["weights"].detach().clone(),
    "derivatives": dataset["derivatives"].detach().clone(),
})
gen_model = IndicatorTraining(
    layers=gen_layers, eta=0.05, alpha=20, friction=friction, options=gen_options
)
gen_datamodule = DictModule(dataset_gen, lengths=[0.8, 0.2], batch_size=512, shuffle=True)
gen_metrics = MetricsCallback()
gen_early_stop = EarlyStopping(monitor="train_loss", min_delta=1e-4, patience=500, verbose=False)
gen_trainer = lightning.Trainer(
    callbacks=[gen_metrics, gen_early_stop],
    max_epochs=100,
    enable_checkpointing=False,
    logger=False,
    limit_val_batches=0,
    num_sanity_val_steps=0,
    accelerator="gpu",
    devices=1,
)
gen_model = gen_model.cuda()
gen_trainer.fit(gen_model, gen_datamodule)
# Plot training metrics
fig, ax = plt.subplots(1, 1, figsize=(4, 3))
ax = plot_metrics(gen_metrics.metrics,
                  keys=["train_loss", "train_loss_var", "train_loss_ortho"],
                  colors=["fessa1", "fessa3", "fessa4"],
                  ax=ax, yscale="log")
plt.close(fig)
# Compute eigenfunctions
gen_model = gen_model.to(device)
dataset_gen_dev = DictDataset({
    "data": dataset_gen["data"].to(device),
    "weights": dataset_gen["weights"].to(device),
    "derivatives": dataset_gen["derivatives"].to(device),
})
g, evals, evecs = gen_model.compute_eigenfunctions(dataset_gen_dev, recompute=True)
coeffs = evecs.cpu().detach().real
print(f"  Eigenvalues: {evals.cpu().detach().numpy()}")
# Create trivial model and save standalone TorchScript
trivial = IndicatorProduction(
    layers=gen_layers,
    eta=0.05,
    alpha=20,
    friction=friction.cpu(),
    coeffs=coeffs[:, 0],
).to("cpu").to(torch.float32)

# Copy NN weights from trained model
trivial.nn = copy.deepcopy(gen_model.nn).to("cpu").to(torch.float32)
bias_model = IndicatorBiasModel(trivial.to("cpu"), l=1, e=1e-7)
desc_cpu = dataset_gen["data"].cpu().detach().float()
bias_vals = bias_model(desc_cpu).detach()
bias_range = (bias_vals.max() - bias_vals.min()).item()
lambda_value = 200.0 / bias_range if bias_range > 0 else 1.0
peak_value = lambda_value * bias_vals.max().item()
print(f"  lambda_value = {lambda_value:.4f}  (normalize bias range to 40)")
print(f"  peak_bias    = {peak_value:.4f}")

gen_model = gen_model.to("cpu")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type          | Params | Mode  | FLOPs | In sizes | Out sizes
---------------------------------------------------------------------------------
0 | loss_fn | GeneratorLoss | 1      | train | 0     | ?        | ?        
1 | nn      | FeedForward   | 2.4 K  | train | 4.7 K | [1, 40]  | [1, 1]   
---------------------------------------------------------------------------------
2.4 K     Trainable params
0         Non-trainable params
2.4 K     Total params
0.010     Total estimated model params size (MB)
8         Modules in train mode
0         Modules in eval mode
4.7 K     Total Flops


Epoch 0:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 99: 100%|██████████| 1/1 [00:00<00:00, 46.05it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 99: 100%|██████████| 1/1 [00:00<00:00, 42.04it/s]
  Eigenvalues: [-8.672103e-05]
  lambda_value = 311.3848  (normalize bias range to 40)
  peak_bias    = -2315.3065


In [ ]:
## Export helper functions for NPT and graph support
## Needs to be integrated in the future in the mlcolvar library
from torch import nn


def make_lightning_traceable(module: nn.Module):
    """Make Lightning modules traceable by JIT by setting dummy trainer.

    PyTorch JIT's trace_module uses hasattr() to check for exported methods,
    which triggers LightningModule.trainer property causing "not attached to Trainer" error.
    This function sets a dummy _trainer attribute to prevent that error.

    Applies recursively to all submodules.
    """
    # Apply to root module
    if hasattr(module, '_trainer') or hasattr(module.__class__, 'trainer'):
        try:
            _ = module.trainer  # Test if it causes error
        except Exception:
            module._trainer = object()  # Set dummy to prevent error

    # Apply recursively to all submodules
    for name, submodule in module.named_modules():
        if submodule is module:
            continue
        if hasattr(submodule, '_trainer') or hasattr(submodule.__class__, 'trainer'):
            try:
                _ = submodule.trainer
            except Exception:
                submodule._trainer = object()

    return module

class FlattenGraphPreprocessNPT(nn.Module):
    """
    GNN wrapper for NPT simulations that accepts box as a second input.
    
    This wrapper is designed for deployment in NPT simulations where the
    box size changes at each step. The box is passed from PLUMED at runtime.
    
    PLUMED centers coordinates around origin (-L/2 to L/2), but the model
    was trained on LAMMPS-style coordinates (0 to L). This wrapper automatically
    computes and applies the coordinate shift from the box diagonal at runtime.
    
    Model signature: forward(positions[1, N*3], box[1, 9]) -> features
    
    Parameters
    ----------
    gnn : nn.Module
        The GNN model (e.g., JacobianNormalizedGNN wrapping GNNTransformerDescriptor)
    n_nodes : int
        Number of atoms
    feat_dim : int
        Feature dimension per node (typically 3)
    """
    
    def __init__(
        self, 
        gnn: nn.Module, 
        n_nodes: int, 
        feat_dim: int,
    ):
        super().__init__()
        self.gnn = gnn
        self.n_nodes = int(n_nodes)
        self.feat_dim = int(feat_dim)
    
    def _box_flat_to_matrix(self, box_flat: torch.Tensor) -> torch.Tensor:
        """Convert flattened 9-element box to 3x3 matrix.
        
        Parameters
        ----------
        box_flat : torch.Tensor
            Shape (1, 9) or (9,) - flattened 3x3 cell matrix (row-major)
            
        Returns
        -------
        torch.Tensor
            Shape (3, 3) cell matrix
        """
        if box_flat.dim() == 2:
            box_flat = box_flat.squeeze(0)  # (9,)
        return box_flat.view(3, 3)
    
    def _compute_half_box(self, cell: torch.Tensor) -> torch.Tensor:
        """Compute half box dimensions from cell matrix diagonal.
        
        For orthorhombic boxes, this returns [Lx/2, Ly/2, Lz/2].
        For triclinic boxes, uses the diagonal elements as approximation.
        
        Parameters
        ----------
        cell : torch.Tensor
            Shape (3, 3) cell matrix
            
        Returns
        -------
        torch.Tensor
            Shape (3,) half box dimensions
        """
        # Extract diagonal: [cell[0,0], cell[1,1], cell[2,2]]
        # Use device-aware constant to avoid TorchScript issues
        two = torch.tensor(2.0, device=cell.device, dtype=cell.dtype)
        return torch.diag(cell) / two
    
    def forward(self, x: torch.Tensor, box: torch.Tensor) -> torch.Tensor:
        """
        Forward pass with runtime box for NPT simulations.
        
        Automatically shifts coordinates by half-box to convert from PLUMED's
        centered coordinates (-L/2 to L/2) to LAMMPS-style (0 to L).
        
        Parameters
        ----------
        x : torch.Tensor
            Flat positions (1, N*3) or (N*3,) from PLUMED (centered at origin)
        box : torch.Tensor
            Flattened box (1, 9) or (9,) - 3x3 cell matrix in row-major order
            
        Returns
        -------
        torch.Tensor
            GNN output features
        """
        # Handle input shape
        if x.dim() == 1:
            x = x.view(1, -1)
        B, D = x.shape
        expected = self.n_nodes * self.feat_dim
        assert D == expected, f"Expected flattened size {expected}, got {D}"
        
        # Convert box to cell matrix
        cell = self._box_flat_to_matrix(box)  # (3, 3)
        
        # Compute half-box shift from box diagonal (dynamic for NPT)
        # PLUMED centers coords at origin, LAMMPS uses 0 to L
        half_box = self._compute_half_box(cell)  # (3,)
        
        # Reshape positions to (B, N, 3) for per-dimension shift
        x_nodes = x.view(B, self.n_nodes, self.feat_dim)  # (B, N, 3)
        
        # Apply coordinate shift: add half_box to each atom's xyz
        # half_box is (3,), broadcasts over (B, N, 3)
        x_nodes = x_nodes + half_box
        
        # Call GNN with runtime cell for NPT
        out = self.gnn(x_nodes, cell=cell)
        
        if out.dim() == 2:
            return out
        elif out.dim() == 3 and out.size(1) == 1:
            return out.squeeze(1)
        else:
            return out.view(B, -1)


class CommittorNPTWrapper(nn.Module):
    """
    Committor wrapper for NPT simulations that accepts box as second input.
    
    Model signature: forward(positions[1, N*3], box[1, 9]) -> CV[1, 1]
    
    This is designed for use with PYTORCH_MODEL_BIAS_VERLET_BOX which reads
    the box from PLUMED's getBox() and passes it to the model.
    """
    
    def __init__(self, committor: nn.Module, preprocess: nn.Module):
        super().__init__()
        self.comm = committor
        self.pre = preprocess
        
        # Clear preprocessing in committor (we handle it)
        if hasattr(self.comm, 'preprocessing'):
            self.comm.preprocessing = None
        if hasattr(self.comm, 'postprocessing') and self.comm.postprocessing is not None:
            self.comm.postprocessing = None
        if hasattr(self.comm, 'sigmoid') and self.comm.sigmoid is not None:
            self.comm.sigmoid = None
        
    def forward(self, x: torch.Tensor, box: torch.Tensor) -> torch.Tensor:
        """
        Forward pass with runtime box for NPT.
        
        Parameters
        ----------
        x : torch.Tensor
            Flat positions (1, N*3)
        box : torch.Tensor
            Flattened box (1, 9) - 3x3 cell matrix in row-major order
            
        Returns
        -------
        torch.Tensor
            CV output (1, 1)
        """
        # Preprocessing with runtime box
        feats = self.pre(x, box)
        
        # Committor forward
        out = self.comm.forward_cv(feats) if hasattr(self.comm, 'forward_cv') else self.comm(feats)
        return out
    
def export_indicator_npt(
    generator_trivial: nn.Module,
    gnn: nn.Module,
    out_path: str,
    n_nodes: int,
    feat_dim: int = 3,
    device: str = "cpu",
    example_input: Optional[torch.Tensor] = None,
    example_box: Optional[torch.Tensor] = None,
) -> torch.jit.ScriptModule:
    """Export a GeneratorTrivial wrapped with NPT GNN preprocessing for PLUMED.

    Produces a model with signature:
        forward(positions[1, N*3], box[1, 9]) -> Tensor[1, 1]

    The model uses the same FlattenGraphPreprocessNPT + CommittorNPTWrapper
    infrastructure as the committor, replacing the committor's forward_cv with
    the generator trivial model's forward_cv = exp(-nn(x)).

    Parameters
    ----------
    generator_trivial : nn.Module
        Trained GeneratorTrivial instance (on any device; will be deep-copied to `device`).
    gnn : nn.Module
        Normalized GNN on CPU (will be deep-copied to `device`).
    out_path : str
        Output .pt file path.
    n_nodes : int
        Number of atoms.
    feat_dim : int
        Spatial dimensions per atom (3).
    device : str
        Tracing device ('cpu' recommended for PLUMED compatibility).
    example_input : torch.Tensor, optional
        Example positions (1, N*3) for tracing.
    example_box : torch.Tensor, optional
        Example box (1, 9) for tracing.

    Returns
    -------
    torch.jit.ScriptModule
    """
    dev = torch.device(device)

    # Deep-copy and move to device
    gen_triv = copy.deepcopy(generator_trivial).to(dev).eval()
    gnn_dev = copy.deepcopy(gnn).to(dev).eval()

    # Make all Lightning submodules traceable
    make_lightning_traceable(gen_triv)
    make_lightning_traceable(gnn_dev)

    # Preprocessing wrapper: positions + box -> GNN descriptors
    flat_pre = FlattenGraphPreprocessNPT(gnn=gnn_dev, n_nodes=n_nodes, feat_dim=feat_dim).to(dev)
    make_lightning_traceable(flat_pre)

    # CommittorNPTWrapper calls self.comm.forward_cv(feats); GeneratorTrivial has forward_cv
    wrapper = CommittorNPTWrapper(gen_triv, flat_pre).to(dev).eval()
    make_lightning_traceable(wrapper)

    # Force all parameters to correct device
    for name, param in wrapper.named_parameters():
        if param.device != dev:
            param.data = param.data.to(dev)

    # Prepare example inputs
    if example_input is not None:
        dummy_pos = example_input.to(dev).view(1, -1)
    else:
        dummy_pos = torch.randn(1, n_nodes * feat_dim, dtype=torch.float32, device=dev)
    if example_box is not None:
        dummy_box = example_box.to(dev).view(1, 9)
    else:
        dummy_box = (torch.eye(3, device=dev) * 10.0).view(1, 9)

    # Test before tracing
    with torch.no_grad():
        test_out = wrapper(dummy_pos, dummy_box)
    print(f"  Test CV value: {test_out[0, 0].item():.6f}")

    # Trace and save
    ts = torch.jit.trace(wrapper, (dummy_pos, dummy_box), strict=False)
    torch.jit.save(ts, out_path)
    print(f"  Saved: {out_path}")
    return ts

In [ ]:
normalized_gnn_cpu = copy.deepcopy(model_normalized).cpu()
# Use a REAL sample from the dataset for tracing/scripting
example_pos = ds_unbiased['data'][0:1].cpu()  # Take first sample

# Get example box from dataset bounds (for NPT tracing)
# The bounds are in LAMMPS format: [xlo, xhi, ylo, yhi, zlo, zhi]
example_bounds = unbiased_bounds[0]  # First sample's bounds
# Convert to cell matrix: orthorhombic box [[Lx,0,0], [0,Ly,0], [0,0,Lz]]
Lx = example_bounds[1] - example_bounds[0]
Ly = example_bounds[3] - example_bounds[2]
Lz = example_bounds[5] - example_bounds[4]
example_box = torch.tensor([
    [Lx, 0, 0],
    [0, Ly, 0],
    [0, 0, Lz]
], dtype=torch.float32).view(1, 9)  # Flatten to (1, 9) for model input

# Get the GNN cutoff for Verlet list
gnn_cutoff = normalized_gnn_cpu.gnn.cutoff if hasattr(normalized_gnn_cpu.gnn, 'cutoff') else 5.0

export_indicator_npt(
        generator_trivial=trivial,
        gnn=normalized_gnn_cpu,
        out_path="../test_data/exported_indicator_npt.pt",
        n_nodes=559,
        feat_dim=3,
        device="cuda",
        example_input=example_pos,
        example_box=example_box,
    )